<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/06-Function-Calling-%26-Data_Extraction/03_External_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import inspect, json
import requests

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

In [7]:
def get_completion(prompt, model="llama-3.3-70b-versatile"):
  messages = [{'role':'system','content': "You are being used as a tool calling agent, just respond with the function call ONLY"},
              {"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [8]:
def get_completion_from_messages(messages, tools,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0, ):
    response = client.chat.completions.create(
        model=model,
        tools=tools,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.tool_calls[0]

In [19]:
def build_prompt(function_list, user_query):
  f_prompt = ""
  for function in function_list:
    signature = inspect.signature(function)
    docstring = function.__doc__
    prompt= f''' Function:
def {function.__name__}{signature}
  """
  {docstring.strip()}
  """
     '''
    f_prompt += prompt
  f_prompt+= f"User Query: {user_query}<human_end>"
  return f_prompt

In [31]:
def give_joke( category : str):
  """
  Joke Categories. Supports: Any, Misc, Programming, Dark, Pun, Spooky, Christmas
  """
  url = f"https://v2.jokeapi.dev/joke/{category}"
  joke = requests.get(url)
  if joke.json()["type"] == "single":
    print(joke.json()["joke"])
  else:
      print(joke.json()["setup"])
      print(joke.json()["delivery"])


In [32]:
query = "Hey! Can you get me a joke for this December?"
prompt= build_prompt([give_joke],query)
call = get_completion(prompt)

In [33]:
exec(call)

What's Santa's favourite type of music?
Wrap!


In [35]:
query = "Hey! Can you get me a joke for coding?"
prompt= build_prompt([give_joke],query)
call = get_completion(prompt)
exec(call)

Why did the web developer walk out of a resturant in disgust?
The seating was laid out in tables.
